## Wczytanie danych i podział na train valid test

In [1]:
import os
import random
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
# --- 1. Globalne ziarno losowości ---
SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# --- 2. Wczytanie i złączenie danych ---
trans = pd.read_csv("dane/train_transaction.csv")
ident = pd.read_csv("dane/train_identity.csv")

df = trans.merge(ident, on="TransactionID", how="left")

# --- 3. Podział Out-Of-Time (chronologiczny) ---
df = df.sort_values("TransactionDT").reset_index(drop=True)

n = len(df)
n_train = int(n * 0.60)
n_valid = int(n * 0.80)   # 60% + 20%

train = df.iloc[:n_train]
valid = df.iloc[n_train:n_valid]
test  = df.iloc[n_valid:]

# --- 4. Weryfikacja podziału ---
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} | shape: {part.shape} | fraud rate: {part['isFraud'].mean():.4f}")


train | shape: (354324, 434) | fraud rate: 0.0338
valid | shape: (118108, 434) | fraud rate: 0.0390
test  | shape: (118108, 434) | fraud rate: 0.0344


## Data preprocessing

In [2]:
# Wymaga sklearn >= 1.3 (TargetEncoder) i scipy
import joblib
from scipy.stats import skew
from sklearn.feature_selection import VarianceThreshold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder,
                                    TargetEncoder, FunctionTransformer)
from sklearn.impute import SimpleImputer

# ── Stałe i grupy kolumn ─────────────────────────────────────────────────────
TARGET         = 'isFraud'
MISSING_THRESH = 0.80   # dla nie-V: odrzuć kolumny z NaN > 80%
V_CORR_THRESH  = 0.05   # dla V_xxx: zachowaj jeśli |corr(isFraud)| ≥ 0.05
VAR_THRESH     = 0.01   # odrzuć cechy numeryczne z wariancją < 0.01
SKEW_THRESH    = 3      # signed_log1p tylko dla |skewness| > 3
CARD_THRESH    = 15     # granica cat_low / cat_high

# Wykrywamy grupy kolumn ze struktury surowych danych
M_COLS  = [c for c in train.columns if c.startswith('M') and c[1:].isdigit()]
D_COLS  = [c for c in train.columns if c.startswith('D') and c[1:].isdigit()]
ID_COLS = [c for c in train.columns
           if c.startswith('id_') or c in ('DeviceType', 'DeviceInfo')]

# ── 1. Feature Engineering ───────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # Czas z TransactionDT
    df['hour']      = (df['TransactionDT'] // 3600) % 24
    df['dayofweek'] = (df['TransactionDT'] // (3600 * 24)) % 7

    # Flaga: email nadawcy == email odbiorcy (fraud rate 3x wyższy gdy True wg EDA)
    df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(np.int64)

    # M1–M9: T→1, F→0, NaN→-1
    # NaN to odrębny stan semantyczny (niezweryfikowane), nie brakująca wartość
    for c in M_COLS:
        if c in df.columns:
            df[c] = df[c].map({'T': 1, 'F': 0}).fillna(-1).astype(np.float64)

    # Flagi missingness dla bloków D i identity:
    # sam brak wartości koreluje z fraudem (np. D7: 14.9% vs 2.7% wg EDA)
    for c in D_COLS + ID_COLS:
        if c in df.columns:
            df[f'{c}_isnan'] = df[c].isna().astype(np.int64)

    return df.drop(columns=['TransactionID', 'TransactionDT', TARGET], errors='ignore')

X_train = engineer_features(train);  y_train = train[TARGET].reset_index(drop=True)
X_valid = engineer_features(valid);  y_valid = valid[TARGET].reset_index(drop=True)
X_test  = engineer_features(test);   y_test  = test[TARGET].reset_index(drop=True)

# ── 2. Filtracja kolumn ──────────────────────────────────────────────────────
isnan_cols   = [c for c in X_train.columns if c.endswith('_isnan')]
v_cols       = [c for c in X_train.columns if c.startswith('V') and c[1:].isdigit()]
missing_rate = X_train.isna().mean()

# V_xxx: zachowaj jeśli NaN < 80% LUB |korelacja z isFraud| ≥ 0.05
# Ślepa filtracja 80% odrzucałaby 23 kolumny V o korelacji 0.05–0.28 (wg EDA)
v_corr = X_train[v_cols].corrwith(y_train).abs()
v_keep = [c for c in v_cols
          if missing_rate[c] < MISSING_THRESH or v_corr.get(c, 0) >= V_CORR_THRESH]

# Pozostałe (bez V_xxx i flag _isnan): standardowy próg 80%
# _isnan flagi zawsze zachowujemy – mają 0 NaN i niosą sygnał
other_cols = [c for c in X_train.columns if c not in v_cols + isnan_cols]
other_keep = [c for c in other_cols if missing_rate.get(c, 0) < MISSING_THRESH]

keep_cols = v_keep + other_keep + isnan_cols
X_train, X_valid, X_test = X_train[keep_cols], X_valid[keep_cols], X_test[keep_cols]
print(f"Kolumny po filtracji: {len(keep_cols)} "
      f"(V_xxx: {len(v_keep)}, inne: {len(other_keep)}, _isnan: {len(isnan_cols)})")

# ── 3. Grupy zmiennych, wariancja, skośność ──────────────────────────────────
# "Stable" = kolumny celowo wykluczone z VarianceThreshold i signed_log1p:
#   _isnan: binarne 0/1, semantycznie istotne nawet przy niskiej wariancji
#   M_COLS: pre-zakodowane {-1, 0, 1} – nie mają sensu logować
stable = set(isnan_cols) | {c for c in M_COLS if c in X_train.columns}

num_all = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_all = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

# Filtr wariancji (tylko nie-stable)
num_for_filter = [c for c in num_all if c not in stable]
selector = VarianceThreshold(threshold=VAR_THRESH)
selector.fit(X_train[num_for_filter])
low_var  = [c for c, ok in zip(num_for_filter, selector.get_support()) if not ok]

X_train = X_train.drop(columns=low_var)
X_valid = X_valid.drop(columns=low_var)
X_test  = X_test.drop(columns=low_var)
num_all = [c for c in num_all if c not in low_var]
print(f"Usunięto {len(low_var)} cech o niskiej wariancji → {len(num_all)} cech num")

# Skośność (tylko nie-stable) → signed_log1p kompresuje ogon bez wykrzywiania pozostałych
skewness   = X_train[[c for c in num_all if c not in stable]].apply(lambda c: skew(c.dropna()))
num_skew   = skewness[skewness.abs() > SKEW_THRESH].index.tolist()
num_normal = [c for c in num_all if c not in num_skew]  # zawiera stable

cat_low  = [c for c in cat_all if X_train[c].nunique() < CARD_THRESH]
cat_high = [c for c in cat_all if X_train[c].nunique() >= CARD_THRESH]

print(f"Numeryczne: {len(num_all)} (skośne: {len(num_skew)}, normalne+stable: {len(num_normal)})")
print(f"Kat. niska kard.: {len(cat_low)}  → {cat_low}")
print(f"Kat. wysoka kard.: {len(cat_high)} → {cat_high}")

# ── 4. ColumnTransformer ─────────────────────────────────────────────────────
def signed_log1p(x):
    # Kompresuje skośne rozkłady; działa dla x ujemnych (w odróżnieniu od log1p)
    return np.sign(x) * np.log1p(np.abs(x))

preprocessor = ColumnTransformer(transformers=[
    ('num_skew', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(signed_log1p, validate=False)),
        ('scl', MinMaxScaler()),
    ]), num_skew),
    ('num_normal', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', MinMaxScaler()),
    ]), num_normal),
    ('cat_low', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary')),
    ]), cat_low),
    ('cat_high', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('enc', TargetEncoder(target_type='binary', smooth='auto')),  # kategoria → P(fraud)
        ('scl', MinMaxScaler()),
    ]), cat_high),
], remainder='drop')

# ── 5. Fit & Transform (fit tylko na train) ──────────────────────────────────
preprocessor.fit(X_train, y_train)

X_train_proc = preprocessor.transform(X_train).astype(np.float32)
X_valid_proc  = preprocessor.transform(X_valid).astype(np.float32)
X_test_proc   = preprocessor.transform(X_test).astype(np.float32)

assert not np.isnan(X_train_proc).any(), "NaN w X_train_proc!"
assert not np.isnan(X_valid_proc).any(),  "NaN w X_valid_proc!"
assert not np.isnan(X_test_proc).any(),   "NaN w X_test_proc!"

# ── 6. Filtrowanie pod Autoenkoder ───────────────────────────────────────────
# Sieć uczy się rekonstruować TYLKO normalne transakcje (y == 0).
# Fraudy będą miały wyższy błąd rekonstrukcji → próg detekcji anomalii.
X_train_ae = X_train_proc[y_train.values == 0]
X_valid_ae  = X_valid_proc   # pełny zbiór – ground truth: y_valid / y_test
X_test_ae   = X_test_proc

print(f"\nGotowe macierze:")
print(f"X_train_ae : {X_train_ae.shape}  ← tylko normalne transakcje")
print(f"X_valid_ae : {X_valid_ae.shape}")
print(f"X_test_ae  : {X_test_ae.shape}")
print(f"dtype      : {X_train_ae.dtype}")

# ── 7. Zapis ─────────────────────────────────────────────────────────────────
joblib.dump(preprocessor,             'preprocessor_ae.pkl')
joblib.dump(X_train.columns.tolist(), 'input_cols_ae.pkl')  # finalna lista kolumn wejściowych
print("\nZapisano: preprocessor_ae.pkl, input_cols_ae.pkl")


c:\Users\matma\Documents\SGH\SEM II\Data mining\Projekt_DM\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\matma\Documents\SGH\SEM II\Data mining\Projekt_DM\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Kolumny po filtracji: 444 (V_xxx: 321, inne: 68, _isnan: 55)
Usunięto 25 cech o niskiej wariancji → 402 cech num
Numeryczne: 402 (skośne: 247, normalne+stable: 155)
Kat. niska kard.: 13  → ['ProductCD', 'card4', 'card6', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']
Kat. wysoka kard.: 4 → ['P_emaildomain', 'R_emaildomain', 'id_31', 'DeviceInfo']

Gotowe macierze:
X_train_ae : (342336, 451)  ← tylko normalne transakcje
X_valid_ae : (118108, 451)
X_test_ae  : (118108, 451)
dtype      : float32

Zapisano: preprocessor_ae.pkl, input_cols_ae.pkl


## Trening modelu AutoEncoder

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# ── 1. DataLoadery ────────────────────────────────────────────────────────────
BATCH_SIZE = 512

X_train_t = torch.tensor(X_train_ae, dtype=torch.float32)
X_valid_t  = torch.tensor(X_valid_ae,  dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid_t),  batch_size=BATCH_SIZE, shuffle=False)

# ── 2. Architektura ───────────────────────────────────────────────────────────
INPUT_DIM  = 451
BOTTLENECK = 32

class FraudAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        act  = nn.LeakyReLU(0.1)
        drop = nn.Dropout(0.2)

        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), act, drop,  # dropout na 256
            nn.Linear(256, 128),       act,          # bez dropout – tuż przed bottleneck
            nn.Linear(128, BOTTLENECK), act,
        )
        self.decoder = nn.Sequential(
            nn.Linear(BOTTLENECK, 128), act, drop,  # dropout na 128
            nn.Linear(128, 256),        act, drop,  # dropout na 256
            nn.Linear(256, INPUT_DIM),              # bez dropout na wyjściu
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# ── 3. Konfiguracja ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Urządzenie: {device}")

model     = FraudAutoEncoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ── 4. Trening z Early Stopping ───────────────────────────────────────────────
MAX_EPOCHS = 100
PATIENCE   = 10

best_val_loss     = float('inf')
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # Trening
    model.train()
    train_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(device)
        recon = model(batch)
        loss  = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch)
    train_loss /= len(X_train_t)

    # Walidacja
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (batch,) in valid_loader:
            batch = batch.to(device)
            val_loss += criterion(model(batch), batch).item() * len(batch)
    val_loss /= len(X_valid_t)

    print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | train={train_loss:.6f} | val={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_ae_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping po {epoch} epokach (patience={PATIENCE}).")
            break

print(f"\nNajlepszy val_loss: {best_val_loss:.6f}")

# ── 5. Błędy rekonstrukcji na zbiorze walidacyjnym ────────────────────────────
model.load_state_dict(torch.load('best_ae_model.pth', weights_only=True))
model.eval()

recon_errors = []
with torch.no_grad():
    for (batch,) in valid_loader:
        batch = batch.to(device)
        # MSE per obserwacja (średnia po 451 cechach)
        errors = ((model(batch) - batch) ** 2).mean(dim=1)
        recon_errors.append(errors.cpu().numpy())

valid_reconstruction_errors = np.concatenate(recon_errors)
print(f"valid_reconstruction_errors: shape={valid_reconstruction_errors.shape}, "
      f"min={valid_reconstruction_errors.min():.6f}, max={valid_reconstruction_errors.max():.6f}")


Urządzenie: cuda
Epoch   1/100 | train=0.017588 | val=0.007670
Epoch   2/100 | train=0.006937 | val=0.004963
Epoch   3/100 | train=0.004842 | val=0.003787
Epoch   4/100 | train=0.004013 | val=0.003191
Epoch   5/100 | train=0.003522 | val=0.002770
Epoch   6/100 | train=0.003154 | val=0.002412
Epoch   7/100 | train=0.002885 | val=0.002200
Epoch   8/100 | train=0.002688 | val=0.002062
Epoch   9/100 | train=0.002530 | val=0.001924
Epoch  10/100 | train=0.002381 | val=0.001794
Epoch  11/100 | train=0.002253 | val=0.001711
Epoch  12/100 | train=0.002133 | val=0.001575
Epoch  13/100 | train=0.002032 | val=0.001480
Epoch  14/100 | train=0.001939 | val=0.001418
Epoch  15/100 | train=0.001868 | val=0.001359
Epoch  16/100 | train=0.001810 | val=0.001310
Epoch  17/100 | train=0.001751 | val=0.001268
Epoch  18/100 | train=0.001706 | val=0.001252
Epoch  19/100 | train=0.001664 | val=0.001218
Epoch  20/100 | train=0.001624 | val=0.001199
Epoch  21/100 | train=0.001589 | val=0.001166
Epoch  22/100 | t

In [8]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
import numpy as np

fraud_errors  = valid_reconstruction_errors[y_valid == 1]
normal_errors = valid_reconstruction_errors[y_valid == 0]

print(f"Normalne — mediana: {np.median(normal_errors):.6f} | 95p: {np.percentile(normal_errors, 95):.6f} | 99p: {np.percentile(normal_errors, 99):.6f}")
print(f"Fraudy   — mediana: {np.median(fraud_errors):.6f} | 95p: {np.percentile(fraud_errors, 95):.6f} | 99p: {np.percentile(fraud_errors, 99):.6f}")

# Szukaj progu maksymalizującego F1
thresholds = np.linspace(valid_reconstruction_errors.min(), valid_reconstruction_errors.max(), 1000)
f1_scores  = [f1_score(y_valid, valid_reconstruction_errors >= t) for t in thresholds]

best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nNajlepszy próg:  {best_threshold:.6f}")
print(f"F1 (walidacja):  {f1_scores[best_idx]:.4f}")
print(f"Precision:       {precision_score(y_valid, valid_reconstruction_errors >= best_threshold):.4f}")
print(f"Recall:          {recall_score(y_valid, valid_reconstruction_errors >= best_threshold):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"AUPRC:    {average_precision_score(y_valid, valid_reconstruction_errors):.4f}")


Normalne — mediana: 0.000338 | 95p: 0.002481 | 99p: 0.004644
Fraudy   — mediana: 0.001486 | 95p: 0.009195 | 99p: 0.014891

Najlepszy próg:  0.003156
F1 (walidacja):  0.3073
Precision:       0.3204
Recall:          0.2952
ROC AUC:  0.7735
AUPRC:    0.2360


## Te wyniki są bardzo niezadowalające, próbujemy dalej

In [11]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
import numpy as np

# Przelicz błędy rekonstrukcji trzema metodami
errors_mean  = []
errors_max   = []
errors_top10 = []

model.eval()
with torch.no_grad():
    for (batch,) in valid_loader:
        batch = batch.to(device)
        diff2 = (model(batch) - batch) ** 2

        errors_mean.append(diff2.mean(dim=1).cpu().numpy())
        errors_max.append(diff2.max(dim=1)[0].cpu().numpy())
        errors_top10.append(diff2.topk(10, dim=1)[0].mean(dim=1).cpu().numpy())

errors_mean  = np.concatenate(errors_mean)
errors_max   = np.concatenate(errors_max)
errors_top10 = np.concatenate(errors_top10)

def best_threshold_f1(errors, y_true, n=1000):
    thresholds = np.linspace(errors.min(), errors.max(), n)
    f1_scores  = [f1_score(y_true, errors >= t) for t in thresholds]
    idx = np.argmax(f1_scores)
    return thresholds[idx], f1_scores[idx]

def evaluate(name, errors, y_true):
    threshold, _ = best_threshold_f1(errors, y_true)
    preds = errors >= threshold
    print(f"\n── {name} ──")
    print(f"  Próg:      {threshold:.6f}")
    print(f"  F1:        {f1_score(y_true, preds):.4f}")
    print(f"  Precision: {precision_score(y_true, preds):.4f}")
    print(f"  Recall:    {recall_score(y_true, preds):.4f}")

evaluate("mean(dim=1)  [baseline]", errors_mean,  y_valid)
evaluate("max(dim=1)",              errors_max,    y_valid)
evaluate("topk(10).mean(dim=1)",   errors_top10,  y_valid)
print(f"ROC AUC:  {roc_auc_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"AUPRC:    {average_precision_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"ROC AUC (max):    {roc_auc_score(y_valid, errors_max):.4f}")
print(f"ROC AUC (topk10): {roc_auc_score(y_valid, errors_top10):.4f}")



── mean(dim=1)  [baseline] ──
  Próg:      0.003156
  F1:        0.3073
  Precision: 0.3204
  Recall:    0.2952

── max(dim=1) ──
  Próg:      0.172790
  F1:        0.1963
  Precision: 0.1378
  Recall:    0.3407

── topk(10).mean(dim=1) ──
  Próg:      0.064286
  F1:        0.2513
  Precision: 0.1922
  Recall:    0.3626
ROC AUC:  0.7735
AUPRC:    0.2360
ROC AUC (max):    0.7443
ROC AUC (topk10): 0.7617


In [12]:
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
import numpy as np

def extract_latent(data_array, batch_size=512):
    tensor = torch.tensor(data_array, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=False)
    latents = []
    model.eval()
    with torch.no_grad():
        for (batch,) in loader:
            latents.append(model.encoder(batch.to(device)).cpu().numpy())
    return np.concatenate(latents)

print("Wyciąganie reprezentacji z bottleneck...")
latent_train = extract_latent(X_train_ae)
latent_valid = extract_latent(X_valid_ae)
print(f"latent_train: {latent_train.shape}")
print(f"latent_valid: {latent_valid.shape}")

# Trening IsolationForest na normalnych transakcjach (latent space)
print("\nTrening IsolationForest...")
iso = IsolationForest(n_estimators=200, contamination='auto', random_state=42, n_jobs=-1)
iso.fit(latent_train)

# Predykcja domyślna (-1 = anomalia, 1 = normalne)
preds_default = (iso.predict(latent_valid) == -1).astype(int)

print("\n── Latent Space + IsolationForest (domyślny próg) ──")
print(f"  F1:        {f1_score(y_valid, preds_default):.4f}")
print(f"  Precision: {precision_score(y_valid, preds_default):.4f}")
print(f"  Recall:    {recall_score(y_valid, preds_default):.4f}")

# Optymalny próg na anomaly score
scores = -iso.score_samples(latent_valid)  # wyższy = bardziej anomalny
thresholds = np.linspace(scores.min(), scores.max(), 1000)
f1_list    = [f1_score(y_valid, scores >= t) for t in thresholds]
best_idx   = np.argmax(f1_list)
best_preds = scores >= thresholds[best_idx]

print("\n── Latent Space + IsolationForest (optymalny próg) ──")
print(f"  Próg:      {thresholds[best_idx]:.6f}")
print(f"  F1:        {f1_list[best_idx]:.4f}")
print(f"  Precision: {precision_score(y_valid, best_preds):.4f}")
print(f"  Recall:    {recall_score(y_valid, best_preds):.4f}")

print(f"ROC AUC:  {roc_auc_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"AUPRC:    {average_precision_score(y_valid, valid_reconstruction_errors):.4f}")
print(f"ROC AUC (latent IF): {roc_auc_score(y_valid, scores):.4f}")


Wyciąganie reprezentacji z bottleneck...
latent_train: (342336, 32)
latent_valid: (118108, 32)

Trening IsolationForest...

── Latent Space + IsolationForest (domyślny próg) ──
  F1:        0.1730
  Precision: 0.1250
  Recall:    0.2809

── Latent Space + IsolationForest (optymalny próg) ──
  Próg:      0.499053
  F1:        0.1737
  Precision: 0.1244
  Recall:    0.2874
ROC AUC:  0.7735
AUPRC:    0.2360
ROC AUC (latent IF): 0.7225
